# ChromaDB Vector Store & Enterprise Persistence Architecture

## 1. Overview and Core Motivation
**ChromaDB** is the AI-native, open-source embedding database designed for developer simplicity and enterprise persistence:
- **Zero-Configuration Persistence**: Out-of-the-box local storage backed by SQLite and ClickHouse/DuckDB mechanisms.
- **HNSW Indexing**: Uses Hierarchical Navigable Small World graphs for sub-millisecond approximate nearest neighbor lookups.
- **Rich Metadata Querying**: Multi-condition metadata filtering (`$and`, `$or`, `$eq`, `$in`) directly at the vector query level.
- **Complete RAG Lifecycle**: Seamlessly connects loaders, recursive splitters, HuggingFace embeddings, and Groq LLMs.

## 2. Technology Stack
- **Embeddings Model**: HuggingFace `sentence-transformers/all-MiniLM-L6-v2` (384-dimensional dense vectors).
- **Vector Store**: `langchain_community.vectorstores.Chroma`.
- **Language Model**: Groq `openai/gpt-oss-120b` (or `gemma2-9b-it`) via LangChain `init_chat_model`.

---

## 3. Architecture & Vector Retrieval Flow

![Vector Stores Architecture](workflow_vectorstores.png)

In [23]:
# Step 1: Load environment variables and API keys
import os
import warnings
warnings.filterwarnings('ignore') # Suppress integration deprecation notices

from dotenv import load_dotenv, find_dotenv

# Search for .env in current and parent directories
load_dotenv('../.env')
load_dotenv(find_dotenv())

# Verify Groq API key is configured
groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    print(f"GROQ_API_KEY detected successfully: {groq_key[:6]}...{groq_key[-4:]}")
else:
    print("Warning: GROQ_API_KEY not found in environment. Please set it in your .env file.")

GROQ_API_KEY detected successfully: gsk_Fh...AC3v


In [24]:
## create sample documents
# Here we define three domain documents covering fundamental AI concepts:
sample_docs = [
"""
Machine Learning Fundamentals
Machine learning is a subset of artificial intelligence that enables systems to learn
and improve from experience without being explicitly programmed. There are three main
types of machine learning: supervised learning, unsupervised learning, and reinforcement
learning. Supervised learning uses labeled data to train models, while unsupervised
learning finds patterns in unlabeled data. Reinforcement learning learns through
interaction with an environment using rewards and penalties.
""",
"""
Deep Learning and Neural Networks
Deep learning is a subset of machine learning based on artificial neural networks.
These networks are inspired by the human brain and consist of layers of interconnected
nodes. Deep learning has revolutionized fields like computer vision, natural language
processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly
effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers
excel at sequential data processing.
""",
"""
Natural Language Processing (NLP)
NLP is a field of AI that focuses on the interaction between computers and human language.
Key tasks in NLP include text classification, named entity recognition, sentiment analysis,
machine translation, and question answering. Modern NLP heavily relies on transformer
architectures like BERT, GPT, and T5. These models use attention mechanisms to understand
context and relationships between words in text.
"""
]

print(f"Defined {len(sample_docs)} sample domain documents.")

Defined 3 sample domain documents.


In [25]:
# Save sample documents to the existing data/text_files directory to simulate a real-world file ingestion workflow
import os

data_dir = "../data/text_files" if os.path.exists("../data/text_files") else "./data/text_files"
os.makedirs(data_dir, exist_ok=True)

for i, doc in enumerate(sample_docs):
    with open(os.path.join(data_dir, f"doc_{i}.txt"), "w", encoding="utf-8") as f:
        f.write(doc.strip())

print(f"Sample docs created in : {data_dir}")

Sample docs created in : ../data/text_files


### 1. Document Loading
In a production RAG pipeline, documents come from heterogeneous sources (PDFs, Markdown files, databases, S3 buckets).
LangChain's **`DirectoryLoader`** combined with **`TextLoader`** scans a target directory, extracts raw text, and attaches provenance metadata (such as file path).

In [26]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader

# Ingest all .txt documents from the directory
loader = DirectoryLoader(
    data_dir, 
    glob="*.txt", 
    loader_cls=TextLoader, 
    loader_kwargs={"encoding": "utf-8"}, 
    show_progress=True
)

documents = loader.load()

# print(f"Loaded {len(documents)} documents.")
# for i, doc in enumerate(documents):
#     print(f"Document {i}: {doc.metadata['source']}")
#     print(f"Content: {doc.page_content[:100]}...")  # Print first 100 characters of content

[print(f"Document {i}: {doc.metadata['source']}\nContent: {doc.page_content[:100]}...") for i, doc in enumerate(documents)]

100%|██████████| 5/5 [00:00<00:00, 792.28it/s]

Document 0: ..\data\text_files\doc_0.txt
Content: Machine Learning Fundamentals
Machine learning is a subset of artificial intelligence that enables s...
Document 1: ..\data\text_files\doc_1.txt
Content: Deep Learning and Neural Networks
Deep learning is a subset of machine learning based on artificial ...
Document 2: ..\data\text_files\doc_2.txt
Content: Natural Language Processing (NLP)
NLP is a field of AI that focuses on the interaction between compu...
Document 3: ..\data\text_files\machine_learning.txt
Content: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...
Document 4: ..\data\text_files\python_intro.txt
Content: Python Programming Introduction

Python is a high-level, interpreted programming language known for ...


[None, None, None, None, None]

### 2. Document Splitting (Chunking)
Why chunk documents?
1. **Context Window Limits**: LLMs have maximum token limits. Passing an entire 100-page book is inefficient and costly.
2. **Retrieval Precision**: Smaller, semantically focused chunks yield much higher similarity scores for specific queries than long, mixed-topic documents.
3. **Chunk Overlap**: Overlapping contiguous chunks (e.g., 50 characters) ensures that concepts falling right on the boundary between two chunks are not severed.

In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize text splitter with chunk_size and chunk_overlap
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # Maximum characters per chunk
    chunk_overlap=50,   # Overlap between consecutive chunks to maintain context
    length_function=len,
    separators=[" "]    # Split on space boundaries to avoid breaking words
)

chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents.")
#show all chunks
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:")
    print(f"Content: {chunk.page_content[:150]}...")
    print(f"Metadata: {chunk.metadata}")

Created 8 chunks from 5 documents.

Chunk 1:
Content: Machine Learning Fundamentals
Machine learning is a subset of artificial intelligence that enables systems to learn
and improve from experience withou...
Metadata: {'source': '..\\data\\text_files\\doc_0.txt'}

Chunk 2:
Content: through
interaction with an environment using rewards and penalties....
Metadata: {'source': '..\\data\\text_files\\doc_0.txt'}

Chunk 3:
Content: Deep Learning and Neural Networks
Deep learning is a subset of machine learning based on artificial neural networks.
These networks are inspired by th...
Metadata: {'source': '..\\data\\text_files\\doc_1.txt'}

Chunk 4:
Content: (RNNs) and Transformers
excel at sequential data processing....
Metadata: {'source': '..\\data\\text_files\\doc_1.txt'}

Chunk 5:
Content: Natural Language Processing (NLP)
NLP is a field of AI that focuses on the interaction between computers and human language.
Key tasks in NLP include ...
Metadata: {'source': '..\\data\\text_files\\doc_

### 3. Dense Vector Embeddings (HuggingFace)
An embedding model converts text into dense numerical vectors in a continuous geometric space where semantically similar phrases are located close to each other.

- **Model**: `sentence-transformers/all-MiniLM-L6-v2`
- **Vector Dimension**: 384 dimensions
- **Efficiency**: Extremely lightweight, fast CPU inference, free and open-source (no API costs).

In [28]:
# Load open-source HuggingFace sentence transformer embeddings
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Embed sample document chunks
texts = [chunk.page_content for chunk in chunks]
embeddings = embedding_model.embed_documents(texts)
print(f"Generated {len(embeddings)} chunk embeddings. Each vector dimension: {len(embeddings[0])}")

# Test embedding a single query
sample_text = "Machine Learning is fascinating"
sample_embedding = embedding_model.embed_query(sample_text)
print(f"Sample query embedding shape: ({len(sample_embedding)},)")
print(f"First 5 vector values: {sample_embedding[:5]}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8608.24it/s]


Generated 8 chunk embeddings. Each vector dimension: 384
Sample query embedding shape: (384,)
First 5 vector values: [-0.04763006046414375, -0.09523380547761917, 0.08622033894062042, 0.008249763399362564, 0.012193558737635612]


### 4. ChromaDB Vector Store Setup & Local Persistence
**ChromaDB** stores chunk embeddings and their corresponding text/metadata.
Under the hood, Chroma uses:
- **HNSW (Hierarchical Navigable Small World)**: In-memory and persisted graph index for high-dimensional vector search and sub-millisecond approximate nearest neighbor (ANN) retrieval.
- **SQLite Database**: Persistent relational storage for document IDs, chunk text strings, metadata dictionaries, and collection configurations on disk (in `./chroma_db`).

In [29]:
from langchain_community.vectorstores import Chroma

persist_dir = "./chroma_db"
collection_name = "rag_collection"

# Clear the previous local collection before rebuilding it (avoids duplicate vectors on re-run)
try:
    old_vectorstore = Chroma(
        persist_directory=persist_dir,
        collection_name=collection_name,
        embedding_function=embedding_model,
    )
    old_vectorstore.delete_collection()
    print("Previous collection cleared.")
except Exception:
    pass

# Create and persist the Chroma vectorstore from document chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_dir,
    collection_name=collection_name,
)

vector_count = len(vectorstore.get(include=[])["ids"])
print(f"Vectorstore created with {vector_count} vectors in collection '{collection_name}'")

Previous collection cleared.
Vectorstore created with 8 vectors in collection 'rag_collection'


### 5. Similarity Search & Understanding Distance Scores

When querying ChromaDB, your search question is embedded using the exact same HuggingFace model and compared against stored vectors.

#### Understanding Distance and Similarity Metrics:
1. **L2 (Squared Euclidean Distance) - ChromaDB Default**:
   $$d(u, v) = \sum_{i=1}^{n} (u_i - v_i)^2$$
   - **Score interpretation**: Lower score means **MORE SIMILAR** (distance closer to 0).
   - A score of **0.0** denotes an exact, identical vector match.
   - Typically values range from 0.0 to 1.5+ for normalized embeddings.

2. **Cosine Similarity / Distance**:
   - Cosine Similarity measures the angle between vectors: $\cos(	heta) = rac{u \cdot v}{\|u\| \|v\|}$.
   - Range: -1.0 to 1.0 (1.0 = identical direction / highest similarity).
   - Cosine Distance is $1 - 	ext{Cosine Similarity}$ (where lower = more similar).

In [30]:
# Test basic similarity search
query = "What is machine learning?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}\n")
for i, result in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"Content: {result.page_content[:180]}...")
    print(f"Metadata: {result.metadata}\n")

Query: What is machine learning?

Result 1:
Content: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It f...
Metadata: {'source': '..\\data\\text_files\\machine_learning.txt'}

Result 2:
Content: Machine Learning Fundamentals
Machine learning is a subset of artificial intelligence that enables systems to learn
and improve from experience without being explicitly programmed....
Metadata: {'source': '..\\data\\text_files\\doc_0.txt'}

Result 3:
Content: Deep Learning and Neural Networks
Deep learning is a subset of machine learning based on artificial neural networks.
These networks are inspired by the human brain and consist of l...
Metadata: {'source': '..\\data\\text_files\\doc_1.txt'}



In [31]:
# Advanced Similarity Search with numerical distance scores
query = "What is deep learning?"
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

print(f"Query: '{query}'\n")
for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"Result {i} (Distance Score: {score:.4f}):")
    print(f"Content: {doc.page_content[:180]}...")
    print(f"Metadata: {doc.metadata}")
    print("-" * 50)

print("\nNote: Lower score indicates closer geometric proximity in the vector space.")

Query: 'What is deep learning?'

Result 1 (Distance Score: 0.5742):
Content: Deep Learning and Neural Networks
Deep learning is a subset of machine learning based on artificial neural networks.
These networks are inspired by the human brain and consist of l...
Metadata: {'source': '..\\data\\text_files\\doc_1.txt'}
--------------------------------------------------
Result 2 (Distance Score: 1.0379):
Content: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It f...
Metadata: {'source': '..\\data\\text_files\\machine_learning.txt'}
--------------------------------------------------
Result 3 (Distance Score: 1.0526):
Content: Machine Learning Fundamentals
Machine learning is a subset of artificial intelligence that enables systems to learn
and improve from experience without being explicitly programmed....
Metadata: {'source': '..\\data\\text_files\\doc_0.txt'}
--

In [32]:
# Convert vector store to a LangChain Retriever
# The retriever abstracts the search logic so it can be plugged directly into chains
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000022D8EE30510>, search_kwargs={'k': 3})

### 6. Initializing Groq LLM and Prompt Template
We use **Groq** for high-speed inference.
Groq's LPU (Language Processing Unit) architecture allows serving large open models like `openai/gpt-oss-120b` or `gemma2-9b-it` at extreme tokens-per-second throughput.

In [33]:
from langchain.chat_models import init_chat_model

# Initialize Groq LLM using init_chat_model
# Model options: 'groq:openai/gpt-oss-120b', 'groq:llama-3.3-70b-versatile', or 'groq:gemma2-9b-it'
try:
    llm = init_chat_model('groq:openai/gpt-oss-120b')
    test_res = llm.invoke("Reply with 'Ready' in one word.")
    print(f"Groq LLM initialized successfully: {test_res.content}")
except Exception as e:
    print(f"Fallback to groq:llama-3.3-70b-versatile due to: {e}")
    llm = init_chat_model('groq:llama-3.3-70b-versatile')

Groq LLM initialized successfully: Ready


In [34]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

print("System and Human QA prompt template created.")

System and Human QA prompt template created.


# Why we use create_stuff_documents_chain when we have LCEL?

**`create_stuff_documents_chain`** is a helper function in LangChain designed to handle the "stuff" documents pattern.

"Stuffing" is the simplest and most common way to do RAG: you take a list of retrieved `Document` objects, "stuff" (concatenate) them all together into a single string, and insert that string into your prompt's `{context}` variable to send to the LLM.

### Why use it if we can use LCEL directly?

You are completely right—you **can** build everything manually using raw LCEL (like your `simple_rag_chain`), and many developers prefer doing so for total control. However, `create_stuff_documents_chain` exists because it automates boilerplate tasks and provides built-in safety features:

#### 1. Automatic Document Formatting & Joining

In raw LCEL, if your retriever returns a list of LangChain `Document` objects, you have to write a custom helper function (like `format_docs`) to extract `.page_content` from every document and join them with newlines:

```python
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
```

`create_stuff_documents_chain` handles this conversion automatically behind the scenes. It expects a list of `Document` objects directly, extracts their text, and stitches them together.

#### 2. Advanced Document Prompts (Metadata handling)

With raw LCEL, formatting metadata (like page numbers, authors, or source URLs) alongside text requires writing complex mapping logic. `create_stuff_documents_chain` accepts a `document_prompt` parameter, allowing you to format *each individual document* uniformly (e.g., `<source>Page 5</source>\ncontent...`) before stuffing them into the final prompt.

#### 3. Seamless Integration with Higher-Level Chains

LangChain provides helper functions like `create_retrieval_chain`, which wires a retriever and a document-combining chain together instantly:

```python
from langchain.chains import create_retrieval_chain

# Automatically links retriever output to the stuff documents chain input
retrieval_chain = create_retrieval_chain(retriever, create_stuff_documents_chain(llm, prompt))
```

Using `create_stuff_documents_chain` makes your code plug-and-play with these higher-level orchestration tools, which handle context passing and chat history injection out of the box.

### Summary

* **Use raw LCEL** if you want maximum transparency, minimal abstraction, and absolute control over how data flows.
* **Use `create_stuff_documents_chain`** if you want to write less boilerplate code, seamlessly process lists of `Document` objects, and easily hook into LangChain's pre-built RAG architectures.

In [ ]:
# Testing create_stuff_documents_chain independently
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm, prompt)

query = "What is deep learning?"
retrieved_docs = retriever.invoke(query)
print(f"Retrieved {len(retrieved_docs)} documents for query: '{query}'")
# print(f"Retrieved documents: {[doc.page_content[:100] + '...' for doc in retrieved_docs]}")  # Print first 100 characters of each retrieved document
print("Retrieved documents:\n" + "\n".join(doc.page_content[:100] + "..." for doc in retrieved_docs))

# Manually invoke the document chain with context and question input
response = document_chain.invoke({
    "context": retrieved_docs,
    "input": query,
})

print("\nResponse from document_chain:")
print(response)

Retrieved 3 documents for query: 'What is deep learning?'
Retrieved documents:
Deep Learning and Neural Networks
Deep learning is a subset of machine learning based on artificial ...
Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...
Machine Learning Fundamentals
Machine learning is a subset of artificial intelligence that enables s...

Response from document_chain:
Deep learning is a branch of machine learning that uses artificial neural networks—structures of layered, interconnected nodes inspired by the human brain—to learn directly from data. These networks automatically extract hierarchical features, enabling breakthroughs in areas such as computer vision, natural language processing, and speech recognition. Convolutional Neural Networks, Recurrent Neural Networks, and Transformers are common deep‑learning architectures for different data types.


# Why do we need create_retrieval_chain?
The `create_retrieval_chain` acts as an automated wrapper that handles the manual steps above.

When you pass it the user's input, it automatically:
1. Takes the user's question.
2. Sends that question to the retriever to fetch the relevant `Document` objects.
3. Packages both the original question and the newly retrieved documents into a dictionary.
4. Passes that complete dictionary to the `document_chain` to generate the final answer.

In [36]:
try:
    from langchain.chains import create_retrieval_chain
except ImportError:
    from langchain_classic.chains import create_retrieval_chain

# This chain handles the "generation" part of RAG. It takes a list of raw Document objects, extracts their text, stuffs that text into the {context} placeholder, and sends the final formatted prompt to the LLM.
document_chain = create_stuff_documents_chain(llm, prompt) 

# This chain orchestrates the entire workflow. It links your retriever (the vector database searcher) to your document_chain.
retrieval_chain = create_retrieval_chain(retriever, document_chain)

# Invoke the full end-to-end RAG chain
response = retrieval_chain.invoke({"input": "What is deep learning?"})

# Print just the answer text string
print("Answer:")
print(response["answer"])

Answer:
Deep learning is a subset of machine learning that uses artificial neural networks—structures inspired by the human brain composed of multiple layers of interconnected nodes. These deep neural networks automatically learn hierarchical representations from large amounts of data, enabling breakthroughs in areas like computer vision, natural language processing, and speech recognition.


In [37]:
# Function to query the modern RAG system
def query_rag_modern(question):
    print(f"Question: {question}")
    print("-" * 50)

    # Run the retrieval chain
    result = retrieval_chain.invoke({"input": question})

    # Print answer
    print(f"Answer: {result['answer']}")

    # Print retrieved context
    print("\nRetrieved Context:")
    for i, doc in enumerate(result['context'], 1):
        print(f"\n--- Source {i} ---")
        print(doc.page_content[:200] + "...")

    return result


# Test queries
test_questions = [
    "What are the three types of machine learning?",
    "What is deep learning and how does it relate to neural networks?",
    "What are CNNs best used for?"
]

for question in test_questions:
    result = query_rag_modern(question)
    print("\n" + "=" * 80 + "\n")

Question: What are the three types of machine learning?
--------------------------------------------------
Answer: The three main types of machine learning are **supervised learning** (using labeled data), **unsupervised learning** (discovering patterns in unlabeled data), and **reinforcement learning** (learning through interaction with rewards and penalties).

Retrieved Context:

--- Source 1 ---
Machine Learning Fundamentals
Machine learning is a subset of artificial intelligence that enables systems to learn
and improve from experience without being explicitly programmed. There are three mai...

--- Source 2 ---
Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing...

--- Source 3 ---
through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems...


Question: What is deep learning 

### 7. Alternative RAG Architecture: LangChain Expression Language (LCEL)

#### What is LCEL?
**LangChain Expression Language (LCEL)** is a declarative orchestration syntax that uses Python's pipe operator (`|`) to compose unified, streaming-ready runnables.

#### LCEL Pipeline Flow:
1. **Parallel Input Branching**: User input query is piped simultaneously to the retriever and a `RunnablePassthrough()`.
2. **Context Formatting**: Retrieved document chunks are formatted into a single text context block using a helper function.
3. **Prompt Composition**: `ChatPromptTemplate` merges the formatted context and original user query.
4. **LLM Inference**: The prompt value is fed directly into the Groq LLM (`openai/gpt-oss-120b`).
5. **Output Parsing**: `StrOutputParser` extracts a clean natural language string from the resulting `AIMessage`.

In [ ]:
# Building the RAG chain using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Helper function: Format retrieved documents into a single text block
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Define custom prompt template for LCEL
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question concisely.
If you don't know the answer based on the context, say that you don't know.

Context: {context}

Question: {question}

Answer:""")

# Build the LCEL pipe
rag_chain_lcel = (
    {
        "context": retriever | format_docs,    # Step 1a: Retrieve docs and format to text
        "question": RunnablePassthrough()       # Step 1b: Pass user query through directly
    }
    | custom_prompt                             # Step 2: Feed dictionary into prompt
    | llm                                       # Step 3: Call Groq LLM
    | StrOutputParser()                         # Step 4: Parse AIMessage to clean string
)

print("LCEL RAG chain constructed successfully!")

# Invoke the LCEL chain directly with a string input
response_lcel = rag_chain_lcel.invoke("What is Deep Learning?")
print(f"\nLCEL Answer:\n{response_lcel}")

LCEL RAG chain constructed successfully!

LCEL Answer:
Deep Learning is a subset of machine learning that uses artificial neural networks—layered structures of interconnected nodes inspired by the human brain—to learn from data.


In [39]:
# Reusable query function using the LCEL approach
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # 1. Invoke chain directly
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer:\n{answer}")
    
    # 2. Inspect source documents retrieved
    docs = retriever.invoke(question)
    print(f"\nRetrieved {len(docs)} Source Documents:")
    for i, doc in enumerate(docs, 1):
        print(f"--- Chunk {i} ---")
        print(doc.page_content[:150] + "...")
    
    return answer

# Test LCEL chain on queries
print("Testing LCEL Chain Queries:\n")
query_rag_lcel("What is machine learning?")
print("\n" + "=" * 80 + "\n")
query_rag_lcel("What are CNNs and Transformers used for?")

Testing LCEL Chain Queries:

Question: What is machine learning?
--------------------------------------------------
Answer:
Machine learning is a branch of artificial intelligence that enables systems to learn from data and improve their performance over time without being explicitly programmed. It involves creating programs that can access data, identify patterns, and make decisions based on experience.

Retrieved 3 Source Documents:
--- Chunk 1 ---
Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without bei...
--- Chunk 2 ---
Machine Learning Fundamentals
Machine learning is a subset of artificial intelligence that enables systems to learn
and improve from experience withou...
--- Chunk 3 ---
Deep Learning and Neural Networks
Deep learning is a subset of machine learning based on artificial neural networks.
These networks are inspired by th...


Question: What are CNNs and Transformers used for

'CNNs are used for image‑related tasks such as image processing and recognition, while Transformers are used for handling sequential data, such as language, speech, and other time‑series inputs.'

### 8. Dynamic Document Ingestion (Updating an Existing Vector Store)

In enterprise vector databases, knowledge is rarely static. New manuals, logs, or policies are continuously added.
Instead of wiping and rebuilding the entire ChromaDB collection from scratch, we can dynamically split and add new documents using `vectorstore.add_documents()`.

In [40]:
from langchain_core.documents import Document

# Define new knowledge to add dynamically to ChromaDB
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a specialized branch of machine learning where an agent learns 
to make sequential decisions by interacting with an environment. The agent receives feedback 
in the form of numerical rewards or penalties. Key concepts in RL include:
- States: The current situation of the agent in the environment.
- Actions: The set of all possible moves the agent can make.
- Rewards: Numerical feedback signal evaluating the action taken.
- Policy: The agent's strategy mapping states to actions.
- Value Function: The expected long-term cumulative reward.

Popular algorithms include Q-Learning, Deep Q-Networks (DQN), and Proximal Policy Optimization (PPO).
"""

# Wrap in a LangChain Document with rich metadata
new_doc = Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "reinforcement_learning"}
)

# Chunk the new document using our existing splitter
new_chunks = splitter.split_documents([new_doc])
print(f"Created {len(new_chunks)} chunks from new document.")

# Record vector count before insertion
count_before = len(vectorstore.get(include=[])["ids"])

# Add new document chunks dynamically to the existing Chroma collection
vectorstore.add_documents(new_chunks)

count_after = len(vectorstore.get(include=[])["ids"])
print(f"Total vectors in collection before: {count_before}")
print(f"Total vectors in collection after:  {count_after} (Added {len(new_chunks)} new chunks)")

Created 2 chunks from new document.
Total vectors in collection before: 8
Total vectors in collection after:  10 (Added 2 new chunks)


In [41]:
# Query the RAG system on the newly ingested knowledge
new_question = "What are the key concepts in reinforcement learning?"
print(f"Querying Updated Vector Store: '{new_question}'\n")

updated_result = query_rag_lcel(new_question)

Querying Updated Vector Store: 'What are the key concepts in reinforcement learning?'

Question: What are the key concepts in reinforcement learning?
--------------------------------------------------
Answer:
The key concepts in reinforcement learning are:

- **States** – the current situation of the agent in the environment.  
- **Actions** – the set of possible moves the agent can take.  
- **Rewards** – numerical feedback evaluating each action.  
- **Policy** – the agent’s strategy that maps states to actions.  
- **Value Function** – the expected long‑term cumulative reward.

Retrieved 3 Source Documents:
--- Chunk 1 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a specialized branch of machine learning where an agent learns 
to make sequential de...
--- Chunk 2 ---
signal evaluating the action taken.
- Policy: The agent's strategy mapping states to actions.
- Value Function: The expected long-term cumulative rewa...
--- Chunk 3 ---
Machine Learning Fundament

### 9. Advanced RAG: Conversational Memory & Multi-Turn Chat

#### The Core Problem in Conversational RAG:
LLMs and vector retrievers are stateless by default:
- **Turn 1**: User asks: *"What is Machine Learning?"* -> System answers accurately.
- **Turn 2**: User asks: *"What are its main types?"*
- A standard retriever takes *"What are its main types?"* literally without knowing what *"its"* refers to, retrieving irrelevant chunks.

#### The 2-Step Solution (History-Aware Retrieval):
1. **Query Reformulation**: Before searching ChromaDB, a lightweight LLM call reformulates the latest question into a **standalone query** using the past chat history (e.g., *"What are the main types of machine learning?"*).
2. **Context-Aware QA**: ChromaDB retrieves documents using this standalone query, and the final generation prompt takes both the chat history and the retrieved context to produce an accurate multi-turn answer.

In [ ]:
# Setting up Conversational RAG components
from langchain_classic.chains import create_history_aware_retriever

from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Step 1: Prompt to reformulate the user's latest question using chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Create history-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, 
    retriever, 
    contextualize_q_prompt
)

print("History-aware retriever created successfully!")

History-aware retriever created successfully!


In [43]:
# Step 2: Create QA Document Chain that accepts chat history
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context:
{context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Combine into document chain
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Create the final Conversational RAG chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever, 
    question_answer_chain
)

print("Conversational RAG chain assembled successfully!")

Conversational RAG chain assembled successfully!


In [44]:
# Multi-Turn Conversation Demonstration

# Initialize an empty conversation history
chat_history = []

print("=== CONVERSATION TURN 1 ===")
question_1 = "What is machine learning?"
print(f"User: {question_1}")

# Invoke chain with turn 1
result_1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": question_1
})

print(f"AI: {result_1['answer']}\n")

# Append Turn 1 interaction to chat_history
chat_history.extend([
    HumanMessage(content=question_1),
    AIMessage(content=result_1['answer'])
])

print("=" * 60)
print("=== CONVERSATION TURN 2 (Follow-up with pronoun 'its') ===")
question_2 = "What are its main types?"  # 'its' refers to machine learning from Turn 1!
print(f"User: {question_2}")

# Invoke chain with turn 2 + chat history
result_2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": question_2
})

print(f"AI: {result_2['answer']}\n")

# Append Turn 2 interaction to chat_history
chat_history.extend([
    HumanMessage(content=question_2),
    AIMessage(content=result_2['answer'])
])

print("=" * 60)
print(f"Total messages maintained in conversation history: {len(chat_history)}")
for i, msg in enumerate(chat_history, 1):
    role = "User" if isinstance(msg, HumanMessage) else "Assistant"
    print(f"[{i}] {role}: {msg.content[:80]}...")

=== CONVERSATION TURN 1 ===
User: What is machine learning?
AI: Machine learning is a branch of artificial intelligence that allows computers to learn from data and improve their performance without being explicitly programmed. It enables systems to identify patterns, make predictions, or take actions based on experience. In essence, it automates the creation of models that adapt as they encounter new information.

=== CONVERSATION TURN 2 (Follow-up with pronoun 'its') ===
User: What are its main types?
AI: The main types of machine learning are:

1. **Supervised learning** – models are trained on labeled data to predict outcomes.  
2. **Unsupervised learning** – models discover patterns or structure in unlabeled data.  
3. **Reinforcement learning** – agents learn by interacting with an environment and receiving rewards or penalties.

Total messages maintained in conversation history: 4
[1] User: What is machine learning?...
[2] Assistant: Machine learning is a branch of artificial in